In [93]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [94]:
data = pd.read_csv(r'D:\Creditcard_fraud_detection\data\processed\creditcard_preprocessed.csv')
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)
data.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:04:08,4767265376804500,"fraud_Stroman, Hudson and Erdman",gas_transport,94.63,Jennifer,Conner,F,4655 David Island,Dublin,PA,18917,40.3750,-75.2045,2158,Transport planner,1961-06-19,189a841a0a8ba03058526bcfe566aab5,1325376248,40.653382,-76.152667,0
1,2019-01-01 00:05:08,6011360759745864,fraud_Corwin-Collins,gas_transport,71.65,Steven,Williams,M,231 Flores Pass Suite 720,Edinburg,VA,22824,38.8432,-78.6003,6018,"Designer, multimedia",1947-08-21,6d294ed2cc447d2c71c7171a3d54967c,1325376308,38.948089,-78.540296,0
2,2019-01-01 00:07:27,5559857416065248,fraud_Kiehn Inc,grocery_pos,96.29,Jack,Hill,M,5916 Susan Bridge Apt. 939,Grenada,CA,96038,41.6125,-122.5258,589,Systems analyst,1945-12-21,413636e759663f264aae1819a4d4f231,1325376447,41.657520,-122.230347,0
3,2019-01-01 00:09:03,3514865930894695,fraud_Beier-Hyatt,shopping_pos,7.77,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,High Rolls Mountain Park,NM,88325,32.9396,-105.8189,899,Naval architect,1967-08-30,8a6293af5ed278dea14448ded2685fea,1325376543,32.863258,-106.520205,0
4,2019-01-01 00:17:40,630441765090,fraud_Pacocha-Bauch,shopping_pos,9.55,Susan,Washington,F,759 Erin Mount Suite 956,May,TX,76857,31.9571,-98.9656,1791,Corporate investment banker,1965-07-26,c4b4daebab8be54cadde4b941244ca53,1325377060,31.626350,-98.610225,0


In [95]:
df = data.copy()

print(df.shape)

(222873, 22)


In [96]:
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"],errors="coerce")

df["dob"] = pd.to_datetime(df["dob"],errors="coerce")

df = df.sort_values("trans_date_trans_time").reset_index(drop=True)

print("Chronologically sorted:",df["trans_date_trans_time"].is_monotonic_increasing)

print(df[["trans_date_trans_time", "dob"]].dtypes)

Chronologically sorted: True
trans_date_trans_time    datetime64[us]
dob                      datetime64[us]
dtype: object


In [97]:
df["transaction_hour"] = df["trans_date_trans_time"].dt.hour
df["transaction_day"] = df["trans_date_trans_time"].dt.day

df["transaction_dayofweek"] = (df["trans_date_trans_time"].dt.dayofweek)
df["transaction_month"] = (df["trans_date_trans_time"].dt.month)

df["transaction_year"] = (df["trans_date_trans_time"].dt.year)

In [98]:
df["hour_sin"] = np.sin(2 * np.pi * df["transaction_hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["transaction_hour"] / 24)

In [99]:
df.shape

(222873, 29)

In [100]:
df["dayofweek_sin"] = np.sin(2 * np.pi * df["transaction_dayofweek"] / 7)
df["dayofweek_cos"] = np.cos(2 * np.pi * df["transaction_dayofweek"] / 7)

In [101]:
df["is_weekend"] = (df["transaction_dayofweek"] >= 5).astype(int)

df["is_night"] = (df["transaction_hour"] < 6).astype(int)

df["is_business_hour"] = ((df["transaction_hour"] >= 9) &(df["transaction_hour"] < 18)).astype(int)

In [102]:
df["age"] = (df["trans_date_trans_time"] - df["dob"]).dt.days / 365.25

In [103]:
df["amt_log"] = np.log1p(df["amt"])

In [104]:
df["amount_bucket"] = pd.cut(df["amt"],
    bins=[-np.inf, 10, 50, 100, 250, 500, 1000, np.inf],
    labels=[
        "very_low",
        "low",
        "medium",
        "medium_high",
        "high",
        "very_high",
        "extreme"
    ]
)

###This tells us where the transaction occurs in the card's history

Transaction 1

Transaction 2

Transaction 3

Transaction 500


In [105]:
df["card_transaction_number"] = (df.groupby("cc_num").cumcount())

###Previous transaction features

previous_transaction_amt,

previous_transaction_time

In [106]:
df["previous_transaction_amt"] = (df.groupby("cc_num")["amt"].shift(1))

df["previous_transaction_time"] = (df.groupby("cc_num")["trans_date_trans_time"].shift(1))

10:00 → transaction
10:03 → transaction
10:05 → transaction
10:07 → transaction

In [107]:
df["time_since_previous_transaction_minutes"] = (df["trans_date_trans_time"]
                                                 .sub(df["previous_transaction_time"]).dt.total_seconds()/ 60)

In [108]:
df["card_merchant_count"] = (df.groupby(["cc_num", "merchant"]).cumcount())

df["is_new_merchant"] = (df["card_merchant_count"] == 0).astype(int)

In [109]:
df["card_city_count"] = ( df.groupby(["cc_num", "city"]).cumcount())

df["is_new_city"] = (df["card_city_count"] == 0).astype(int)

In [110]:
df["card_category_count"] = (df.groupby(["cc_num", "category"]).cumcount())

df["is_new_category"] = (df["card_category_count"] == 0).astype(int)

##Customer–Merchant Distance

In [111]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = ( np.sin(dlat / 2) ** 2+ np.cos(lat1)* np.cos(lat2)* np.sin(dlon / 2) ** 2)
    return R * (2 * np.arcsin(np.sqrt(a)))

In [112]:
df["customer_merchant_distance_km"] = haversine_distance(df["lat"],df["long"],df["merch_lat"],df["merch_long"])

###Previous Transaction Location

In [114]:
df["previous_merch_lat"] = (df.groupby("cc_num")["merch_lat"].shift(1))
df["previous_merch_long"] = (df.groupby("cc_num")["merch_long"].shift(1))

In [115]:
df["distance_from_previous_location_km"] = haversine_distance(
    df["previous_merch_lat"],
    df["previous_merch_long"],
    df["merch_lat"],
    df["merch_long"]
)

In [116]:
df["location_velocity_kmph"] = np.where(
    df["time_since_previous_transaction_minutes"] > 0,
    df["distance_from_previous_location_km"] /
    (df["time_since_previous_transaction_minutes"] / 60),
    np.nan
)

In [117]:
print(df["location_velocity_kmph"].quantile([0.50, 0.90, 0.95, 0.99, 0.999]))

0.500       3.370242
0.900      35.057597
0.950      85.278113
0.990     562.855741
0.999    6834.224149
Name: location_velocity_kmph, dtype: float64


In [118]:
###card's historical spending baseline

In [119]:
df["previous_avg_amount"] = (df.groupby("cc_num")["amt"].transform(lambda x: x.shift(1).expanding().mean()))


In [120]:
df["previous_std_amount"] = (df.groupby("cc_num")["amt"].transform(lambda x: x.shift(1).expanding().std()))

In [121]:
df["amount_vs_historical_avg"] = (df["amt"]/df["previous_avg_amount"].replace(0, np.nan))

In [122]:
df["amount_zscore"] = ((df["amt"]- df["previous_avg_amount"])/df["previous_std_amount"].replace(0, np.nan))

###Handle infinite values

In [123]:
df = df.replace([np.inf, -np.inf],np.nan)

In [124]:
print(df.shape)

print(df.isnull().sum().sort_values(ascending=False).head(20))

print("Infinite values:",np.isinf(df.select_dtypes(include=np.number)).sum().sum())

print("Duplicate rows:",df.duplicated().sum())

print(df["is_fraud"].value_counts())

(222873, 56)
previous_std_amount                        1966
amount_zscore                              1966
previous_merch_long                         983
previous_merch_lat                          983
previous_transaction_amt                    983
previous_transaction_time                   983
time_since_previous_transaction_minutes     983
previous_avg_amount                         983
distance_from_previous_location_km          983
amount_vs_historical_avg                    983
location_velocity_kmph                      983
street                                        0
lat                                           0
city                                          0
state                                         0
zip                                           0
trans_date_trans_time                         0
cc_num                                        0
merchant                                      0
category                                      0
dtype: int64
Infinite value

In [125]:

df.columns

Index(['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt',
       'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat',
       'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat',
       'merch_long', 'is_fraud', 'transaction_hour', 'transaction_day',
       'transaction_dayofweek', 'transaction_month', 'transaction_year',
       'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'is_weekend',
       'is_night', 'is_business_hour', 'age', 'amt_log', 'amount_bucket',
       'card_transaction_number', 'previous_transaction_amt',
       'previous_transaction_time', 'time_since_previous_transaction_minutes',
       'card_merchant_count', 'is_new_merchant', 'card_city_count',
       'is_new_city', 'card_category_count', 'is_new_category',
       'customer_merchant_distance_km', 'previous_merch_lat',
       'previous_merch_long', 'distance_from_previous_location_km',
       'location_velocity_kmph', 'previous_avg_amount', 'previous_std_a

#target leakage

In [126]:

suspicious_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ["fraud", "target", "label"]
    )
]

print("Potential target-related columns:")
print(suspicious_columns)

Potential target-related columns:
['is_fraud']


In [127]:
# Columns used only for historical feature construction
temporary_columns = [
    "previous_lat",
    "previous_long",
    "previous_transaction_time"
]

df = df.drop(columns=temporary_columns,errors="ignore")

print("Shape after removing temporary columns:", df.shape)

Shape after removing temporary columns: (222873, 55)


In [128]:
identifier_columns = [
    "cc_num",
    "trans_num",
    "Unnamed: 0"
]

df = df.drop(columns=identifier_columns,errors="ignore")

print("Final feature-engineered shape:", df.shape)

Final feature-engineered shape: (222873, 53)


In [129]:
print("Duplicate rows:",df.duplicated().sum())

Duplicate rows: 0


In [130]:
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (
        df.isnull().mean() * 100
    )
})

missing_report = (missing_report.sort_values(
        "missing_percentage",
        ascending=False)
)

missing_report[missing_report["missing_count"] > 0]

,missing_count,missing_percentage
amount_zscore,1966,0.882117
previous_std_amount,1966,0.882117
previous_transaction_amt,983,0.441058
distance_from_previous_location_km,983,0.441058
previous_avg_amount,983,0.441058
previous_merch_long,983,0.441058
previous_merch_lat,983,0.441058
time_since_previous_transaction_minutes,983,0.441058
location_velocity_kmph,983,0.441058
amount_vs_historical_avg,983,0.441058


In [132]:
df.to_csv("../data/processed/creditcard_featured_data.csv",index=False)
print("data is saved now ")
print(df.shape)


data is saved now 
(222873, 53)
